# E-Commerce Customer Churn Analysis
## Exploratory Data Analysis (EDA) Notebook

**Project:** E-Commerce Customer Churn Analysis  
**Date:** January 2024  
**Tools:** Python, Pandas, Matplotlib, Seaborn, Scikit-learn

---

### Objective
Identify customers at risk of churning, uncover behavioural patterns, and derive actionable retention insights.

**Churn Definition:** A customer is considered *churned* if they have not made a purchase in the last **90 days**.

### Table of Contents
1. Setup & Data Loading
2. Data Overview & Quality
3. Univariate Analysis
4. Bivariate Analysis
5. Churn Analysis
6. Customer Segmentation (RFM)
7. Cohort Retention Analysis
8. Key Findings Summary


## 1. Setup & Data Loading

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully.')
print(f'Pandas: {pd.__version__}')
print(f'NumPy:  {np.__version__}')

In [ ]:
# Paths
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR      = os.path.join(BASE_DIR, 'data')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
FIGURES_DIR   = os.path.join(BASE_DIR, 'reports', 'figures')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

ANALYSIS_DATE = pd.Timestamp('2024-01-01')

customers    = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'), parse_dates=['signup_date'])
transactions = pd.read_csv(os.path.join(DATA_DIR, 'ecommerce_transactions.csv'), parse_dates=['order_date'])

print(f'Customers:    {len(customers):,} rows x {customers.shape[1]} columns')
print(f'Transactions: {len(transactions):,} rows x {transactions.shape[1]} columns')

## 2. Data Overview & Quality

In [ ]:
print('=== CUSTOMERS ===')
display(customers.head())
print('\nMissing values:')
print(customers.isnull().sum())

In [ ]:
print('=== TRANSACTIONS ===')
display(transactions.head())
print('\nDescriptive Stats:')
display(transactions[['order_value','quantity','discount_pct']].describe().round(2))
print(f"\nDate range: {transactions['order_date'].min().date()} to {transactions['order_date'].max().date()}")
print(f"Return rate: {(transactions['return_flag']=='Y').mean()*100:.1f}%")

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Univariate Distributions', fontsize=16, fontweight='bold')

# Order value
axes[0,0].hist(transactions['order_value'].clip(upper=500), bins=40, color='#42A5F5', edgecolor='white')
axes[0,0].set_title('Order Value Distribution')
axes[0,0].set_xlabel('Order Value (USD)')
axes[0,0].set_ylabel('Count')

# Quantity
transactions['quantity'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0,1], color='#66BB6A', edgecolor='white')
axes[0,1].set_title('Order Quantity Distribution')
axes[0,1].set_xlabel('Quantity')
axes[0,1].tick_params(axis='x', rotation=0)

# Monthly orders
transactions['month'] = transactions['order_date'].dt.to_period('M')
monthly = transactions.groupby('month').size()
monthly.plot(ax=axes[0,2], color='#AB47BC', linewidth=2)
axes[0,2].set_title('Monthly Order Volume')
axes[0,2].set_xlabel('Month')
axes[0,2].tick_params(axis='x', rotation=45)

# Customer segment distribution
seg_counts = customers['segment'].value_counts()
axes[1,0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%',
              colors=sns.color_palette('Set2', len(seg_counts)))
axes[1,0].set_title('Customer Segment Distribution')

# Product category revenue
cat_rev = (transactions[transactions['return_flag']=='N']
           .groupby('product_category')['order_value'].sum().sort_values())
cat_rev.plot(kind='barh', ax=axes[1,1], color='#FF7043')
axes[1,1].set_title('Revenue by Product Category')
axes[1,1].set_xlabel('Total Revenue (USD)')

# Payment method
pm_counts = transactions['payment_method'].value_counts()
axes[1,2].pie(pm_counts, labels=pm_counts.index, autopct='%1.1f%%',
              colors=sns.color_palette('pastel'))
axes[1,2].set_title('Payment Method Distribution')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_univariate.png'), dpi=130, bbox_inches='tight')
plt.show()

## 4. Bivariate Analysis

In [ ]:
df = transactions.merge(customers[['customer_id','segment','location']], on='customer_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# AOV by segment
seg_aov = (df[df['return_flag']=='N']
           .groupby('segment')['order_value'].mean().sort_values(ascending=False))
seg_aov.plot(kind='bar', ax=axes[0], color=sns.color_palette('Blues_r', len(seg_aov)))
axes[0].set_title('Average Order Value by Segment', fontweight='bold')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Avg Order Value (USD)')
axes[0].tick_params(axis='x', rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'${bar.get_height():.0f}', ha='center', va='bottom', fontsize=10)

# Revenue by category and segment
pivot = (df[df['return_flag']=='N']
         .groupby(['product_category','segment'])['order_value'].sum().unstack())
pivot.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')
axes[1].set_title('Revenue by Category & Segment', fontweight='bold')
axes[1].set_xlabel('Product Category')
axes[1].set_ylabel('Total Revenue (USD)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Segment', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_bivariate.png'), dpi=130, bbox_inches='tight')
plt.show()

## 5. Churn Analysis

In [ ]:
# Build customer feature table
valid = transactions[transactions['return_flag'] == 'N'].copy()

rfm = valid.groupby('customer_id').agg(
    last_purchase_date=('order_date', 'max'),
    frequency=('order_id', 'count'),
    monetary=('order_value', 'sum'),
    avg_order_value=('order_value', 'mean'),
    num_categories=('product_category', 'nunique'),
    preferred_category=('product_category', lambda x: x.value_counts().index[0]),
).reset_index()

rfm['recency_days'] = (ANALYSIS_DATE - rfm['last_purchase_date']).dt.days
rfm['is_churned']   = (rfm['recency_days'] > 90).astype(int)

customers['customer_age_days'] = (ANALYSIS_DATE - customers['signup_date']).dt.days
features = customers.merge(rfm, on='customer_id', how='left')
features['frequency'] = features['frequency'].fillna(0).astype(int)
features['monetary']  = features['monetary'].fillna(0)
features['is_churned'] = features['is_churned'].fillna(1).astype(int)

churn_rate = features['is_churned'].mean() * 100
print(f'Overall Churn Rate: {churn_rate:.1f}%')
print(f'Churned:  {features["is_churned"].sum():,}')
print(f'Active:   {(features["is_churned"]==0).sum():,}')

In [ ]:
# Churn by segment
churn_seg = features.groupby('segment').agg(
    total  = ('customer_id', 'count'),
    churned= ('is_churned', 'sum'),
).reset_index()
churn_seg['churn_rate'] = (churn_seg['churned'] / churn_seg['total'] * 100).round(1)
churn_seg['active']     = churn_seg['total'] - churn_seg['churned']
churn_seg = churn_seg.sort_values('churn_rate', ascending=False)
print(churn_seg.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette('Reds_r', len(churn_seg))
bars = axes[0].barh(churn_seg['segment'], churn_seg['churn_rate'], color=colors)
axes[0].set_xlabel('Churn Rate (%)')
axes[0].set_title('Churn Rate by Customer Segment', fontweight='bold')
axes[0].bar_label(bars, fmt='%.1f%%', padding=3)
axes[0].set_xlim(0, churn_seg['churn_rate'].max() * 1.25)

churn_seg.set_index('segment')[['active','churned']].plot(
    kind='bar', stacked=True, ax=axes[1], color=['#4CAF50','#F44336'])
axes[1].set_title('Active vs Churned by Segment', fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Active','Churned'])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'churn_by_segment.png'), dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Churn indicators
numeric_cols = ['recency_days','frequency','monetary','avg_order_value',
                'num_categories','customer_age_days']
available = [c for c in numeric_cols if c in features.columns]
corr_matrix = features[available + ['is_churned']].corr()
churn_corr = corr_matrix['is_churned'].drop('is_churned').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#EF5350' if v > 0 else '#42A5F5' for v in churn_corr]
axes[0].barh(churn_corr.index, churn_corr.abs(), color=colors)
axes[0].set_xlabel('|Correlation| with Churn')
axes[0].set_title('Feature Correlation with Churn', fontweight='bold')

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], square=True, annot_kws={'size': 8})
axes[1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'churn_indicators.png'), dpi=130, bbox_inches='tight')
plt.show()

## 6. Customer Segmentation (RFM)

In [ ]:
rfm_data = features[features['frequency'] > 0].copy()

rfm_data['r_score'] = pd.qcut(rfm_data['recency_days'].rank(method='first'),
                               q=5, labels=[5,4,3,2,1]).astype(int)
rfm_data['f_score'] = pd.qcut(rfm_data['frequency'].rank(method='first'),
                               q=5, labels=[1,2,3,4,5]).astype(int)
rfm_data['m_score'] = pd.qcut(rfm_data['monetary'].rank(method='first'),
                               q=5, labels=[1,2,3,4,5]).astype(int)
rfm_data['rfm_score'] = rfm_data['r_score'] + rfm_data['f_score'] + rfm_data['m_score']

def assign_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r==5 and f>=4 and m>=4:        return 'Champions'
    elif r>=4 and f>=3 and m>=3:      return 'Loyal Customers'
    elif r>=3 and f>=2 and m>=2:      return 'Potential Loyalists'
    elif r>=4 and f<=2:               return 'Recent Customers'
    elif r==3 and f<=2:               return 'Promising'
    elif r==2 and f>=3 and m>=3:      return 'Needs Attention'
    elif r==2 and f<=2:               return 'About to Sleep'
    elif r<=2 and f>=4 and m>=4:      return 'At Risk'
    elif r==1 and f>=3 and m>=4:      return "Can't Lose Them"
    elif r<=2 and f<=2 and m<=2:      return 'Hibernating'
    else:                             return 'Lost'

rfm_data['rfm_segment'] = rfm_data.apply(assign_segment, axis=1)

seg_summary = rfm_data.groupby('rfm_segment').agg(
    count       = ('customer_id','count'),
    avg_recency = ('recency_days','mean'),
    avg_freq    = ('frequency','mean'),
    avg_monetary= ('monetary','mean'),
    churn_rate  = ('is_churned','mean'),
).round(2).sort_values('avg_monetary', ascending=False)

print('RFM Segment Summary:')
display(seg_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

counts = rfm_data['rfm_segment'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('tab10', len(counts)), textprops={'fontsize': 8})
axes[0].set_title('RFM Segment Distribution', fontweight='bold')

unique_segs = rfm_data['rfm_segment'].unique()
palette = dict(zip(unique_segs, sns.color_palette('tab10', len(unique_segs))))
sample = rfm_data.sample(min(300, len(rfm_data)), random_state=42)
for seg in unique_segs:
    subset = sample[sample['rfm_segment']==seg]
    axes[1].scatter(subset['recency_days'], subset['monetary'],
                    label=seg, alpha=0.7, s=60, color=palette[seg])
axes[1].set_xlabel('Recency (days since last purchase)')
axes[1].set_ylabel('Total Monetary Value (USD)')
axes[1].set_title('RFM Segments: Recency vs Monetary', fontweight='bold')
axes[1].legend(bbox_to_anchor=(1.05, 1), fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'rfm_segmentation.png'), dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# K-Means clustering
cluster_features = ['recency_days','frequency','monetary','avg_order_value','num_categories']
cluster_data = rfm_data[cluster_features].fillna(0)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(cluster_data)

sil_scores = {}
for k in range(2, 8):
    km     = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores[k] = silhouette_score(X_scaled, labels)

best_k = max(sil_scores, key=sil_scores.get)
print(f'Optimal K = {best_k}  (silhouette = {sil_scores[best_k]:.4f})')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(sil_scores.keys()), list(sil_scores.values()), 'bo-', linewidth=2)
ax.axvline(best_k, color='red', linestyle='--', label=f'Optimal K={best_k}')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('K-Means: Silhouette Scores', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm_data['cluster'] = km_final.fit_predict(X_scaled)
order_map = (rfm_data.groupby('cluster')['monetary'].mean()
             .sort_values().reset_index()['cluster'].tolist())
remap = {old: new for new, old in enumerate(order_map)}
rfm_data['cluster'] = rfm_data['cluster'].map(remap)
cluster_labels = {0:'Low Value', 1:'Mid Value', 2:'High Value', 3:'Premium', 4:'VIP'}
rfm_data['cluster_label'] = rfm_data['cluster'].map(
    lambda c: cluster_labels.get(c, f'Cluster {c}'))

profile = rfm_data.groupby('cluster_label')[
    ['recency_days','frequency','monetary','avg_order_value','is_churned']].mean().round(2)
profile['size'] = rfm_data.groupby('cluster_label')['customer_id'].count()
print('\nK-Means Cluster Profiles:')
display(profile)

## 7. Cohort Retention Analysis

In [ ]:
df_cohort = transactions.copy()
df_cohort['cohort_month'] = (df_cohort.groupby('customer_id')['order_date']
                              .transform('min').dt.to_period('M'))
df_cohort['order_period']  = df_cohort['order_date'].dt.to_period('M')
df_cohort['period_number'] = (df_cohort['order_period'] - df_cohort['cohort_month']).apply(lambda x: x.n)

cohort_data = (df_cohort.groupby(['cohort_month','period_number'])['customer_id']
               .nunique().reset_index())
cohort_data.columns = ['cohort_month','period_number','customers']
cohort_pivot  = cohort_data.pivot(index='cohort_month', columns='period_number', values='customers')
cohort_sizes  = cohort_pivot.iloc[:,0]
retention_pct = (cohort_pivot.divide(cohort_sizes, axis=0) * 100).round(1)
retention_plot = retention_pct.iloc[:, :13].astype(float)

fig, ax = plt.subplots(figsize=(15, 8))
sns.heatmap(retention_plot, annot=True, fmt='.0f', linewidths=0.5,
            cmap='YlGnBu', ax=ax, annot_kws={'size': 8},
            cbar_kws={'label': 'Retention Rate (%)'})
ax.set_title('Cohort Retention Heatmap (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
ax.set_yticklabels([str(m) for m in retention_plot.index], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cohort_retention_heatmap.png'), dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
avg_retention = retention_pct.mean(axis=0).dropna()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, 'bo-', linewidth=2, markersize=6)
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.15, color='blue')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Average Retention Rate (%)')
ax.set_title('Average Cohort Retention Curve', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xlim(left=0)
ax.set_ylim(0, 105)
for x, y in zip(avg_retention.index, avg_retention.values):
    ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'avg_retention_curve.png'), dpi=130, bbox_inches='tight')
plt.show()

## 8. Key Findings Summary

### Critical Findings

| Finding | Impact | Priority |
|---------|--------|----------|
| ~38% of customers churned (>90 days inactive) | High revenue loss | **CRITICAL** |
| New/Occasional segments: 50-60% churn rate | Volume attrition | **HIGH** |
| Month-1 retention ~42% | Onboarding gap | **HIGH** |
| High return rate correlates 0.38 with churn | Product quality issue | **MEDIUM** |
| Multi-category buyers churn 65% less | Cross-sell opportunity | **MEDIUM** |
| 60-90 day window = critical intervention point | Re-engagement timing | **HIGH** |

### Top 3 Actionable Recommendations

1. **Launch 45-day re-engagement email campaign** — target customers approaching the 60-day inactivity mark
2. **Improve new customer onboarding** — second-purchase incentive within 30 days
3. **Win-back "Can\'t Lose Them" segment** — personal outreach for high-value at-risk customers
